# RF-DETR Fine-tuning on NTLNP Dataset
### Configured for AWS SageMaker | RFDETRMedium | 17-class wildlife detection
Dataset: NTLNP (25,297 images) — Train 17708 / Valid 5059 / Test 2530

## 1. Environment Setup

In [ ]:
import os

# Set your Roboflow API key.
# Option A: hardcode below (keep this notebook private — do NOT commit to git)
os.environ["ROBOFLOW_API_KEY"] = "YOUR_ROBOFLOW_API_KEY"  # <-- replace this

# Option B: if you stored it as a SageMaker lifecycle env var, comment out A and use:
# os.environ["ROBOFLOW_API_KEY"] = os.environ.get("ROBOFLOW_API_KEY", "")

assert os.environ["ROBOFLOW_API_KEY"] != "YOUR_ROBOFLOW_API_KEY", \
    "Please replace YOUR_ROBOFLOW_API_KEY with your actual key above!"

In [ ]:
# Verify GPU — make sure you launched a GPU-backed SageMaker instance
# Recommended: ml.g4dn.xlarge (T4 16GB)  |  ml.g5.xlarge (A10G 24GB)  |  ml.p3.2xlarge (V100 16GB)
!nvidia-smi

In [ ]:
!pip install -q "rfdetr[train,loggers]>=1.4.0" roboflow "supervision==0.29.1"

## 2. Download NTLNP Dataset from Roboflow

In [ ]:
import roboflow

rf = roboflow.Roboflow(api_key=os.environ["ROBOFLOW_API_KEY"])

# Workspace: deepak-singh-1ojve  |  Project: ntlnp-suhy0  |  Version: 2
project = rf.workspace("deepak-singh-1ojve").project("ntlnp-suhy0")
dataset = project.version(2).download("coco")

print("Dataset downloaded to:", dataset.location)

In [ ]:
# Sanity-check: confirm train/valid/test folders and annotation files exist
# RF-DETR requires each split to have _annotations.coco.json
for split in ["train", "valid", "test"]:
    split_dir = os.path.join(dataset.location, split)
    ann_file  = os.path.join(split_dir, "_annotations.coco.json")
    imgs = [f for f in os.listdir(split_dir) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    ann_ok = os.path.exists(ann_file)
    print(f"{split:6s}: {len(imgs):5d} images  |  _annotations.coco.json: {ann_ok}")
    assert ann_ok, f'Missing annotation file for {split} split!'
print("\nAll splits verified OK.")

## 3. Train RF-DETR Medium

**Adjust `batch_size` / `grad_accum_steps` based on your SageMaker instance (effective batch must = 16):**

| Instance | GPU | VRAM | batch_size | grad_accum_steps |
|---|---|---|---|---|
| ml.g4dn.xlarge | T4 | 16 GB | 4 | 4 |
| ml.g5.xlarge | A10G | 24 GB | 8 | 2 |
| ml.p3.2xlarge | V100 | 16 GB | 4 | 4 |

We run 60 epochs with **early stopping (patience=10)** — training auto-stops once val mAP plateaus, saving you compute cost.

In [ ]:
from rfdetr import RFDETRMedium

# /home/ec2-user/SageMaker/ is persistent across notebook stop/start on SageMaker
# /tmp is NOT persistent — do NOT use it for checkpoints
OUTPUT_DIR = "/home/ec2-user/SageMaker/ntlnp_output"
os.makedirs(OUTPUT_DIR, exist_ok=True)
print("Checkpoints will be saved to:", OUTPUT_DIR)

model = RFDETRMedium()

model.train(
    dataset_dir=dataset.location,
    epochs=60,
    batch_size=4,           # T4/V100 (16GB): 4  |  A10G (24GB): 8
    grad_accum_steps=4,     # effective batch = batch_size * grad_accum_steps = 16
    output_dir=OUTPUT_DIR,
    early_stopping_patience=10,    # stop if no improvement for 10 epochs
    early_stopping_min_delta=0.001,
)

In [ ]:
# List all training artifacts saved to OUTPUT_DIR
for fname in sorted(os.listdir(OUTPUT_DIR)):
    fpath = os.path.join(OUTPUT_DIR, fname)
    size_mb = os.path.getsize(fpath) / 1_000_000
    print(f"{fname:<40s}  {size_mb:7.1f} MB")

In [ ]:
from PIL import Image as PILImage

metrics_path = os.path.join(OUTPUT_DIR, "metrics_plot.png")
if os.path.exists(metrics_path):
    display(PILImage.open(metrics_path))
else:
    print("metrics_plot.png not found — training may still be running")

## 4. Deploy Trained Model to Roboflow

In [ ]:
# Uploads checkpoint_best_total.pth to: deepak-singh-1ojve / ntlnp-suhy0 / version 2
checkpoint_path = os.path.join(OUTPUT_DIR, "checkpoint_best_total.pth")
assert os.path.exists(checkpoint_path), \
    f"Best checkpoint not found at {checkpoint_path} — did training complete?"

model.deploy_to_roboflow(
    workspace="deepak-singh-1ojve",
    project_id="ntlnp-suhy0",
    version=2,
    api_key=os.environ["ROBOFLOW_API_KEY"]
)

## 5. Evaluate the Fine-tuned Model
Clear GPU memory, reload best checkpoint, then run mAP on the test split.

In [ ]:
import gc, torch

del model
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    allocated = torch.cuda.memory_allocated() / 1_000_000
    print(f"GPU memory freed. Allocated now: {allocated:.1f} MB")

In [ ]:
from rfdetr import RFDETRMedium

checkpoint_path = os.path.join(OUTPUT_DIR, "checkpoint_best_total.pth")
assert os.path.exists(checkpoint_path), \
    f"Checkpoint not found: {checkpoint_path}"

model = RFDETRMedium(pretrain_weights=checkpoint_path)
model.optimize_for_inference()
print("Best checkpoint loaded and optimized for inference")

In [ ]:
import supervision as sv

test_img_dir  = os.path.join(dataset.location, "test")
test_ann_file = os.path.join(test_img_dir, "_annotations.coco.json")

assert os.path.exists(test_ann_file), \
    f"Test annotation file not found: {test_ann_file}"

ds = sv.DetectionDataset.from_coco(
    images_directory_path=test_img_dir,
    annotations_path=test_ann_file,
)
print(f"Test set: {len(ds)} images")

In [ ]:
from PIL import Image
from tqdm import tqdm
from supervision.metrics import MeanAveragePrecision

targets, predictions = [], []

for path, image, annotations in tqdm(ds):
    image = Image.open(path)
    detections = model.predict(image, threshold=0)
    targets.append(annotations)
    predictions.append(detections)

map_metric = MeanAveragePrecision()
map_result = map_metric.update(predictions, targets).compute()
print(map_result)

## 6. (Optional) Resume Training if Interrupted
If SageMaker timed out or the kernel died mid-training, run this cell.
`checkpoint.pth` is the full optimizer+scheduler state; it resumes exactly where training stopped.

In [ ]:
# Only run this cell to resume an interrupted training run
from rfdetr import RFDETRMedium

# checkpoint.pth = full state (model + optimizer + scheduler)
# checkpoint_best_total.pth = weights only (restarts optimizer from scratch)
resume_ckpt = os.path.join(OUTPUT_DIR, "checkpoint.pth")
assert os.path.exists(resume_ckpt), \
    f"Resume checkpoint not found: {resume_ckpt}"

model = RFDETRMedium()
model.train(
    dataset_dir=dataset.location,
    epochs=60,
    batch_size=4,
    grad_accum_steps=4,
    output_dir=OUTPUT_DIR,
    resume=resume_ckpt,
    early_stopping_patience=10,
    early_stopping_min_delta=0.001,
)